# Random forests for classification

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
Build hundreds of different trees and let them **vote**. Each tree is trained on a bootstrap sample and with a random subset of features at each node. The trees' errors are partly independent and cancel out in the vote.

$$\hat p(c\mid x)=\frac1B\sum_{b=1}^{B}\hat p_b(c\mid x)$$

## Why is voting strong? (an example)
If each tree is right with probability 0.7, independently, the majority of 25 trees is right with probability about **0.98** (binomial distribution). Real trees are not fully independent, so the gain is smaller; that is why random features (`max_features`) matter.



In [ ]:
from math import comb

# Probability that the majority of n independent voters is right, when each is right with probability p
def majority(n, p):
    return sum(comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n // 2 + 1, n + 1))

for n in [1, 5, 25, 101]:
    print(f"{n:3d} trees -> {majority(n, 0.7):.4f}")



> [One deep tree against a 300-tree forest: the forest's boundary is smoother and more reliable.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/forest_cls.png)

*One deep tree against a 300-tree forest: the forest's boundary is smoother and more reliable.*

[[lab:cls_forest]]

## Hyperparameters
| Parameter | Classification default | Effect |
|---|---|---|
| `n_estimators` | 100 | More = more stable (does not overfit) |
| `max_features` | `"sqrt"` | Fewer = more independent trees |
| `max_depth`, `min_samples_leaf` | none, 1 | Complexity of each tree |
| `class_weight` | None | `"balanced"` for imbalanced data |
| `n_jobs` | None | `-1` = all CPU cores |

## Feature importance
> [The ten most important features for breast cancer diagnosis.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/forest_importance.png)

*The ten most important features for breast cancer diagnosis.*



In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
for name, m in [("one tree", DecisionTreeClassifier(random_state=0)),
                ("forest 300", RandomForestClassifier(300, random_state=0)),
                ("extra trees 300", ExtraTreesClassifier(300, random_state=0))]:
    print(f"{name:16s} accuracy = {cross_val_score(m, X, y, cv=5).mean():.3f}")

rf = RandomForestClassifier(300, oob_score=True, random_state=0).fit(X, y)
print("OOB accuracy:", round(rf.oob_score_, 3))



## Bagging against random forests
| | Bagging | Random forest |
|---|---|---|
| Bootstrap samples | Yes | Yes |
| Random features per node | No | Yes |
| Correlation between trees | Higher | Lower |
| Base model | Any model | Trees only |
| scikit-learn class | `BaggingClassifier` | `RandomForestClassifier` |

> `ExtraTreesClassifier` is even more random: it also picks the cut point at random. It is faster and sometimes more accurate.

### Exercise

Compare the OOB accuracy for max_features 1, 'sqrt' and None.

In [ ]:
# Your answer here